In [8]:
# export_onnx.py
import torch
from repro import seed_everything, configure_torch_threads, collect_meta

from SSD_from_scratch import mySSD

In [9]:
mySSD.forward()

TypeError: mySSD.forward() missing 2 required positional arguments: 'self' and 'x'

In [ ]:
def export_model(model, onnx_path: str, opset: int = 17):
    seed_everything(0, deterministic=True)
    configure_torch_threads(intra_op=1, inter_op=1)  # export should not be “multi-thread nondeterministic”
    model.eval().to("cpu").float()

    dummy = torch.randn(1, 3, 300, 300, dtype=torch.float32)

    torch.onnx.export(
        model,
        dummy,
        onnx_path,
        export_params=True,
        opset_version=opset,
        do_constant_folding=True,
        training=torch.onnx.TrainingMode.EVAL,
        input_names=["images"],
        output_names=["loc", "conf"],
        dynamic_axes={"images": {0: "batch"}, "loc": {0: "batch"}, "conf": {0: "batch"}},
    )

    meta = collect_meta(
        seed=0,
        deterministic=True,
        device="cpu",
        intra=1,
        inter=1,
        onnx_path=onnx_path,
        opset=opset,
    )
    print(meta.to_dict())